# flyct · runs on Kaggle

Rebuilds the FlyWire graph and the saved splits from scratch, then runs the models that need more than a
laptop: cell-type models, and GraphSAGE on a GPU. Results go to `/kaggle/working/runs_kaggle.jsonl`.

In [ ]:
REPO = "scariadon5/flywire-cell-type-prediction"
BRANCH = "main"
METHODS = "sage_conn sage_region sage"            # cell-type partner profile: "profile_linear profile_linear_iter"
TASKS = "super_class cell_class cell_type"        # ... with TASKS = "cell_type"
SPLITS = "all"                                    # or e.g. "frac20 hemi"

try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    token = None  # public repo
url = f"https://{token}@github.com/{REPO}.git" if token else f"https://github.com/{REPO}.git"
%cd /kaggle/working
!rm -rf /kaggle/working/flyct && git clone -q --depth 1 -b {BRANCH} {url} /kaggle/working/flyct
%cd /kaggle/working/flyct
!git log -1 --format='commit %h  %s'

In [ ]:
# Kaggle images already ship pandas, numpy, scipy, scikit-learn, torch and matplotlib.
!pip install -q "duckdb>=1.1" "pyarrow>=17" "numba>=0.60" "lightgbm>=4.5" tqdm
!pip install -q --no-deps -e .

## Data and splits (about 10 minutes; both steps fail loudly if anything differs from the repo)

In [ ]:
!make data PY=python
!make splits PY=python

## Runs

In [ ]:
import torch, os
print("device:", "cuda" if torch.cuda.is_available() else "cpu", "| threads:", torch.get_num_threads())
before = sum(1 for _ in open("results/runs.jsonl"))
!python -m flyct.run --method {METHODS} --task {TASKS} --split {SPLITS}

In [ ]:
lines = open("results/runs.jsonl").read().splitlines()[before:]
open("/kaggle/working/runs_kaggle.jsonl", "w").write("\n".join(lines) + "\n")
print(f"{len(lines)} new runs saved to /kaggle/working/runs_kaggle.jsonl")
!python -m flyct.report